# 🎙️ Hindi TTS — Full GPU Training Pipeline (Google Colab)

**Complete, fixed pipeline** for training a Hindi TTS model on Google Colab using the 26-story Drive dataset.

| Step | What it does |
|------|--------------|
| 1 | Check GPU hardware |
| 2 | Mount Google Drive & verify audio files |
| 3 | Clone repo & install all dependencies |
| 4 | Copy WAV files into repo |
| 5A | Speaker diarization (Pyannote) |
| 5B | Whisper transcription → metadata.csv |
| 6 | Format dataset (GPT-SoVITS + XTTS) |
| 7 | Launch GPT-SoVITS WebUI training |

> **Drive folder:** [hindi_tts_data](https://drive.google.com/drive/folders/17OKMSInQ7-8KFNUn0voAYPhFzViLAFra?usp=sharing)

---
⚠️ **Before running:** Make sure Runtime → Change runtime type → **T4 GPU** is selected.

## ⚡ Step 1: Check GPU & Hardware

In [ ]:
import torch

print('='*55)
print('GPU Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name      :', torch.cuda.get_device_name(0))
    print('VRAM (GB)     :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU')
print('='*55)
!nvidia-smi

## 📁 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_FOLDER = '/content/drive/MyDrive/hindi_tts_data'

if os.path.exists(DRIVE_FOLDER):
    wav_files = [f for f in os.listdir(DRIVE_FOLDER) if f.endswith('.wav')]
    print(f'[+] Drive folder found!')
    print(f'    Total .wav files : {len(wav_files)}')
    for f in wav_files[:5]:
        print(f'    - {f[:70]}')
    if len(wav_files) > 5:
        print(f'    ... and {len(wav_files)-5} more')
else:
    print('[-] Drive folder NOT found at /content/drive/MyDrive/hindi_tts_data')
    print('[*] Trying gdown fallback download...')
    !pip install -q gdown
    import gdown
    DRIVE_FOLDER = '/content/hindi_tts_data'
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    gdown.download_folder(
        'https://drive.google.com/drive/folders/17OKMSInQ7-8KFNUn0voAYPhFzViLAFra?usp=sharing',
        output=DRIVE_FOLDER,
        quiet=False
    )
    wav_files = [f for f in os.listdir(DRIVE_FOLDER) if f.endswith('.wav')]
    print(f'[+] Downloaded {len(wav_files)} .wav files via gdown')

## 📦 Step 3: Clone Repo & Install All Dependencies

In [ ]:
# Clone the repo (skip if already done)
!git clone https://github.com/paarth-error/hindi_tts.git /content/HindiTTS 2>/dev/null || echo '[*] Repo already cloned'

import os
os.chdir('/content/HindiTTS')
print('[+] Working directory:', os.getcwd())

# Install all required packages
# Note: pyannote.audio + demucs are heavy — takes ~3-5 min
!pip install -q \
    edge-tts \
    openai-whisper \
    openai \
    soundfile \
    torchaudio \
    demucs \
    pyannote.audio \
    pyannote.core \
    pydub \
    python-dotenv \
    pandas \
    gdown

print('[+] All dependencies installed!')

## 📂 Step 4: Copy Audio Files Into Pipeline

In [ ]:
import shutil, os

DEST_RAW  = '/content/HindiTTS/data/01_raw_audio'
os.makedirs(DEST_RAW, exist_ok=True)

copied = 0
skipped = 0
for fname in os.listdir(DRIVE_FOLDER):
    if fname.lower().endswith('.wav'):
        src  = os.path.join(DRIVE_FOLDER, fname)
        dest = os.path.join(DEST_RAW, fname)
        if not os.path.exists(dest):
            shutil.copy2(src, dest)
            copied += 1
        else:
            skipped += 1

total = len([f for f in os.listdir(DEST_RAW) if f.endswith('.wav')])
print(f'[+] Copied  : {copied}  new files')
print(f'[+] Skipped : {skipped} already present')
print(f'[+] Total   : {total} WAV files ready in data/01_raw_audio/')

## 🎙️ Step 5A: Vocal Separation (Demucs)

Separates voice from background music/noise in each raw WAV.
Output goes to `data/02_isolated_vocals/`.

> ⏱ ~1–3 min per file on T4 GPU. Skip this step if the Drive folder already contains clean speech-only files.

In [ ]:
!python scripts/02_separate_audio.py

## 🗣️ Step 5B: Speaker Diarization (Pyannote)

Splits each isolated vocal file into per-speaker chunks.
Output: `data/03_diarized_segments/<story>/<SPEAKER_XX>/<chunk>.wav`

> **Requires:** A HuggingFace token with access to `pyannote/speaker-diarization-3.1`.
> Accept the terms at https://huggingface.co/pyannote/speaker-diarization-3.1 first.

In [ ]:
# Paste your HuggingFace token here
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'   # ← replace this

# Write token to .env so the scripts can read it
with open('/content/HindiTTS/.env', 'w') as f:
    f.write(f'HF_TOKEN={HF_TOKEN}\n')

!python scripts/03_diarize_pyannote.py

## 📝 Step 5C: Whisper Transcription → metadata.csv

Transcribes each diarized chunk using Whisper `medium` (Hindi → Devanagari).
Output: `data/metadata.csv`

> ⏱ ~20–40 min for 500–2000 chunks on T4 GPU.

In [ ]:
!python scripts/04_transcribe_whisper.py

## 🔍 Step 5D: Verify Dataset

In [ ]:
import pandas as pd

meta_path = '/content/HindiTTS/data/metadata.csv'
if os.path.exists(meta_path):
    df = pd.read_csv(meta_path)
    print(f'[+] metadata.csv loaded: {len(df)} rows')
    print(f'    Columns  : {list(df.columns)}')
    print(f'    Speakers : {df["speaker_id"].nunique()}')
    print(f'    Stories  : {df["story"].nunique()}')
    print()
    print(df[["speaker_id", "story", "text"]].head(5).to_string(index=False))
else:
    print('[-] metadata.csv not found — did Step 5C finish successfully?')

## 🤖 Step 6: Format Dataset for GPT-SoVITS + XTTS-v2

Reads `data/metadata.csv` (or any existing manifest), resolves audio paths,
and writes:
- `data/07_final/sovits_filelist.txt` → for GPT-SoVITS
- `data/07_final/metadata.csv` + `xtts_config.json` → for Coqui XTTS-v2

In [ ]:
!python scripts/08_format_sovits.py --format sovits
!python scripts/08_format_sovits.py --format xtts

# Show stats
!python scripts/08_format_sovits.py --stats

## 🚀 Step 7: Launch GPT-SoVITS Training WebUI

Clones [RVC-Boss/GPT-SoVITS](https://github.com/RVC-Boss/GPT-SoVITS),
installs its requirements, and launches the training WebUI with a public Gradio share link.

> After running this cell, click the `*.gradio.live` link that appears in the output.
> In the WebUI, point **Training data path** to:
> `/content/HindiTTS/data/07_final/sovits_filelist.txt`

In [ ]:
import os

# Clone GPT-SoVITS
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS 2>/dev/null || echo '[*] Already cloned'

os.chdir('/content/GPT-SoVITS')

# Install GPT-SoVITS dependencies (may take 5–10 min)
!pip install -q -r requirements.txt

# Copy our filelist into the GPT-SoVITS working dir for convenience
!cp /content/HindiTTS/data/07_final/sovits_filelist.txt /content/GPT-SoVITS/hindi_filelist.txt
print('[+] Filelist copied → /content/GPT-SoVITS/hindi_filelist.txt')
print('[+] Launching WebUI with public share link...')
print()

# Launch the WebUI
!python webui.py --share

---

## 🎧 Bonus: Quick Test — Generate Hindi Speech with Edge-TTS

No training needed — instantly test the neural Edge-TTS engine.

In [ ]:
import sys, os
os.chdir('/content/HindiTTS')
sys.path.insert(0, '/content/HindiTTS')

from src.tts_engine import HindiTTSEngine

engine = HindiTTSEngine(output_dir='/content/HindiTTS/outputs')
out = engine.generate_speech(
    text='नमस्ते! यह हिंदी टेक्स्ट-टू-स्पीच का परीक्षण है।',
    filename='colab_test.mp3'
)
print(f'[+] Audio saved: {out}')

# Play inline in Colab
from IPython.display import Audio, display
display(Audio(str(out), autoplay=True))